In [1]:
!pip install transformers torch gradio accelerate

In [2]:
!pip install --upgrade gradio

In [3]:
from transformers import pipeline

# Qwen2.5-0.5B-Instruct: the lighter fallback model from the group's approved
# proposal, chosen here for a faster download and quicker CPU inference.
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

chatbot_pipeline = pipeline(
    "text-generation",
    model=MODEL_NAME,
    dtype="auto",
    device_map="auto",
)

print("Model loaded:", MODEL_NAME)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Model loaded: Qwen/Qwen2.5-0.5B-Instruct


In [11]:
def flatten_content(content):
    """Gradio 6.x can pass message content as a plain string or as a list
    of content parts (e.g. [{"type": "text", "text": "..."}]). This
    normalizes either form into a plain string for the chat template."""
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        parts = []
        for item in content:
            if isinstance(item, dict) and "text" in item:
                parts.append(item["text"])
            elif isinstance(item, str):
                parts.append(item)
        return " ".join(parts)
    return str(content)

def respond(message, history):
    messages = []
    for h in history:
        messages.append({"role": h["role"], "content": flatten_content(h["content"])})
    messages.append({"role": "user", "content": message})

    prompt = chatbot_pipeline.tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

    output = chatbot_pipeline(
        prompt,
        max_new_tokens=200,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        return_full_text=False,
    )

    return output[0]["generated_text"].strip()

In [6]:
import gradio
print(gradio.__version__)

6.24.0


In [12]:
import gradio as gr

demo = gr.ChatInterface(
    fn=respond,
    title="MSAI-631 Group LLM Chatbot",
    description=(
        "A simple question-and-answer chatbot built on Qwen2.5-0.5B-Instruct, "
        "a small open-source language model chosen to run on free, CPU-only "
        "infrastructure. Built for the MSAI-631 Group Project."
    ),
)

demo.launch()

* Running on local URL:  http://127.0.0.1:7864
* To create a public link, set `share=True` in `launch()`.


[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
